# OmegaPDF

Download manhwa panels from [OmegaScans](https://omegascans.org) and generate PDFs using the [OmegaAPI](https://omegaapi.vercel.app).

**How to use:**
1. Run the setup cell below
2. Search for a series or browse popular titles
3. Select a chapter and generate a PDF

In [ ]:
#@title Setup — Install dependencies { display-mode: "form" }
!pip install -q requests Pillow

import requests
import io
import os
from PIL import Image
from IPython.display import display, HTML

BASE_URL = "https://omegaapi.vercel.app"
API = f"{BASE_URL}/api/v1"

def api_get(path, params=None):
    """Make a GET request to the OmegaAPI."""
    r = requests.get(f"{API}{path}", params=params, timeout=30)
    r.raise_for_status()
    return r.json()

print("Ready!")

In [ ]:
#@title Search for a series { display-mode: "form" }
query = "solo leveling"  #@param {type:"string"}

results = api_get(f"/search", params={"q": query})
if results.get("success") and results["data"]:
    for i, s in enumerate(results["data"][:10]):
        chapters = s.get("chaptersCount", "?")
        status = s.get("status", "")
        print(f"{i+1}. {s['title']}  [{status}] — {chapters} chapters  —  slug: {s['slug']}")
else:
    print("No results found. Try a different search term.")

In [ ]:
#@title List chapters for a series { display-mode: "form" }
series_slug = "solo-leveling"  #@param {type:"string"}

series = api_get(f"/series/{series_slug}")
if series.get("success"):
    data = series["data"]
    print(f"Title: {data['title']}")
    print(f"Status: {data.get('status', 'N/A')}")
    print(f"Chapters: {data.get('chaptersCount', len(data.get('chapters', [])))}")
    print("\nAvailable chapters:")
    chapters = data.get("chapters", [])
    for ch in chapters[:30]:
        free = "[Free]" if ch.get("isFree") else "[Paid]"
        print(f"  {ch['name']} {free}  —  slug: {ch['slug']}")
    if len(chapters) > 30:
        print(f"  ... and {len(chapters) - 30} more chapters")
else:
    print(f"Error: {series.get('error', 'Series not found')}")

In [ ]:
#@title Download chapter and generate PDF { display-mode: "form" }
slug = "solo-leveling"  #@param {type:"string"}
chapter = "chapter-1"  #@param {type:"string"}
output_name = "Solo_Leveling_Ch1"  #@param {type:"string"}

# Fetch chapter data
ch_data = api_get(f"/chapter/{slug}/{chapter}")
if not ch_data.get("success"):
    print(f"Error: {ch_data.get('error', 'Chapter not found')}")
else:
    info = ch_data["data"]
    image_urls = info.get("images", [])
    series_title = info.get("series", {}).get("title", slug)
    chapter_name = info.get("name", chapter)
    print(f"Series: {series_title}")
    print(f"Chapter: {chapter_name}")
    print(f"Pages: {len(image_urls)}")

    # Download all panels
    pages = []
    for idx, url in enumerate(image_urls, 1):
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        img = Image.open(io.BytesIO(r.content))
        if img.mode == "RGBA":
            img = img.convert("RGB")
        pages.append(img)
        print(f"  [{idx}/{len(image_urls)}] Panel {idx} downloaded")

    # Build PDF
    pdf_path = f"{output_name}.pdf"
    first = pages[0]
    rest = pages[1:] if len(pages) > 1 else []
    first.save(
        pdf_path,
        format="PDF",
        save_all=True,
        append_images=rest,
        resolution=72,
        quality=85,
    )
    for img in pages:
        img.close()

    size_mb = os.path.getsize(pdf_path) / (1024 * 1024)
    print(f"\nPDF saved: {pdf_path} ({size_mb:.2f} MB, {len(pages)} pages)")
    print("Downloading...")
    from google.colab import files
    files.download(pdf_path)

In [ ]:
#@title Batch download — multiple chapters { display-mode: "form" }
batch_slug = "solo-leveling"  #@param {type:"string"}
start_ch = 1  #@param {type:"integer"}
end_ch = 3  #@param {type:"integer"}

for ch_num in range(start_ch, end_ch + 1):
    ch_id = f"chapter-{ch_num}"
    print(f"\n{'='*40}")
    print(f"Processing {ch_id}...")
    try:
        ch_data = api_get(f"/chapter/{batch_slug}/{ch_id}")
        if not ch_data.get("success"):
            print(f"  Skipping: {ch_data.get('error', 'Not found')}")
            continue

        info = ch_data["data"]
        image_urls = info.get("images", [])
        pages = []
        for url in image_urls:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            img = Image.open(io.BytesIO(r.content))
            if img.mode == "RGBA":
                img = img.convert("RGB")
            pages.append(img)

        pdf_name = f"{batch_slug}_{ch_id}.pdf"
        first = pages[0]
        rest = pages[1:] if len(pages) > 1 else []
        first.save(pdf_name, format="PDF", save_all=True, append_images=rest, resolution=72, quality=85)
        for img in pages:
            img.close()
        print(f"  Saved: {pdf_name} ({len(pages)} pages)")
    except Exception as e:
        print(f"  Error: {e}")

print("\nBatch complete!")